# DAY4 실습 03 — 카이제곱 적합도 검정

In [ ]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from scipy import stats

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "ai4i2020.csv")

df = pd.read_csv(csv_path)

data = df.copy()

## 데이터와 질문

데이터 파일 : `ai4i2020.csv`
출처 : UCI Machine Learning Repository - [AI4I 2020 Predictive Maintenance Dataset](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset)

 한 행은 **운전 기록 1건**이다.

| 열 | 뜻 |
|---|---|
| `Type` | 제품 유형 (L : 저품질, M : 중품질, H : 고품질) |
| `Machine failure` | 고장 여부 (1 : 고장, 0 : 정상) |
| `HDF` | 열 방출 고장 여부 (Heat Dissipation Failure, 1 : 해당) |

**질문** : 고장 기록의 제품 유형 구성은 기준 비율과 같은가?

**실습 전제** : 기준 비율은 학습을 위한 가상 기준이다. 운전 기록끼리 서로 독립이라고 가정한다.

In [ ]:
# 데이터 확인 : 행·열 수, 제품 유형별 건수, 고장 건수
print("행·열:", data.shape)                               # shape : (행 수, 열 수)
print(data["Type"].value_counts())                       # value_counts : 값마다 몇 건인지 센다
print("고장 건수:", data["Machine failure"].sum(), "/ 열 방출 고장 건수:", data["HDF"].sum())
#   두 열 모두 0(아님) · 1(해당)이므로 모두 더하면 1의 개수 = 고장 건수다

행·열: (10000, 14)
Type
L    6000
M    2997
H    1003
Name: count, dtype: int64
고장 건수: 339 / 열 방출 고장 건수: 115


## 1. 카이제곱 검정의 기본 개념

**검정 선택에서의 위치** : 검정은 비교하는 변수의 유형으로 먼저 나뉜다.

| 변수의 유형 | 질문 | 검정 |
|---|---|---|
| 범주형 × 수치형 | 집단에 따라 평균(대표값)이 다른가 | t검정, 일원분산분석 등 |
| 범주형 × 범주형 | 범주의 비율(빈도 분포)이 기준이나 다른 집단과 다른가 | **카이제곱 검정** |
| 수치형 × 수치형 | 두 수치가 함께 변하는가 | 상관분석 |

**카이제곱 검정이 필요한 이유** : 불량 유형, 제품 유형, 고장 여부 같은 범주형 자료는 평균을 낼 수 없다. 대신 범주마다 몇 건인지 센 **빈도**를 비교한다. 카이제곱 검정은 실제로 센 빈도가 기준대로라면 나왔을 빈도와 얼마나 다른지를 하나의 값(χ²)으로 요약하고, 그 차이가 우연으로 생길 수 있는 크기인지 p값으로 판단한다. 수치의 분포를 다루지 않으므로 정규성 · 등분산 확인은 필요 없다.

**관측빈도** : 실제로 센 건수다.

**기대빈도** : 귀무가설이 맞다면 나올 것으로 기대되는 건수다. 관측빈도와 비교하는 기준이 된다.

**카이제곱 통계량** χ² = Σ (관측빈도 O − 기대빈도 E)² ÷ E

- 범주(칸)마다 (O − E)² ÷ E 를 구해 모두 더한 값이다. Σ는 모두 더한다는 뜻이다.
- 칸마다 구한 (O − E)² ÷ E 를 **기여도**라고 부른다. 기여도가 큰 칸이 χ²를 키운 칸, 즉 기대와 가장 많이 어긋난 범주다.
- 관측과 기대가 똑같으면 χ² = 0 이다. 어긋날수록 커지고 p값은 작아진다.

**범주형 변수** : 관측 대상마다 값이 정해진 범주 중 하나로 나뉘는 특성이다(예 : 제품 유형 L · M · H, 고장 여부). 데이터 표에서는 열 하나가 변수 하나다.

**카이제곱 검정 세 가지**

| 구분 | 적합도 검정 | 동질성 검정 | 독립성 검정 |
|---|---|---|---|
| 질문 | 한 집단의 범주별 비율 분포가 기준과 같은가 | 여러 집단의 비율이 서로 같은가 | 두 범주형 변수가 관련이 있는가 |
| 모집단과 범주형 변수 | 모집단 1개 · 범주형 변수 1개 | 모집단 2개 이상 · 범주형 변수 1개 | 모집단 1개 · 범주형 변수 2개 |
| 자료를 모은 방식 | 한 모집단에서 추출 | 집단마다 크기를 정해 추출 | 한 번에 추출한 뒤 두 변수로 분류 |
| 기대빈도 E | 전체 건수 × 기준 비율 | 행 합계 × 열 합계 ÷ 전체 | 행 합계 × 열 합계 ÷ 전체 |
| 자유도 | 범주 수 − 1 | (행 수 − 1) × (열 수 − 1) | (행 수 − 1) × (열 수 − 1) |
| 함수 | `stats.chisquare` | `stats.chi2_contingency` | `stats.chi2_contingency` |



공통 조건 : 관측끼리 독립이고, 기대빈도가 모두 5 이상이다.

공식 문서 : [scipy.stats.chisquare](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chisquare.html) · [scipy.stats.chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)

## 2. 적합도 검정

**적합도 검정**은 한 범주형 변수의 관측 비율이 기준 비율(과거 실적 · 계획 · 이론)과 맞는지 검정한다.

**적용 사례**

| 분야 | 질문 | 기준 비율 |
|---|---|---|
| 품질 관리 | 이번 달 불량 유형 구성이 과거와 같은가 | 과거 불량 유형 비율 |
| 설비 보전 | 고장 기록의 제품 유형 구성이 생산 계획과 같은가 | 생산 계획 비율 |
| 고객 관리 | 이번 분기 문의 유형 구성이 작년과 같은가 | 작년 문의 유형 비율 |

**가설 형식** (예 : 불량 유형 4가지, 과거 비율 40% · 25% · 20% · 15%)

- 귀무가설(H₀): 각 범주의 모비율은 기준 비율과 같다. (p₁ = 0.40, p₂ = 0.25, p₃ = 0.20, p₄ = 0.15)
- 대립가설(H₁): 적어도 한 범주의 모비율이 기준 비율과 다르다.

**진행 순서**

1) 관측빈도 : 범주마다 실제로 센 건수
2) 기준 비율 : 비교할 기준
3) 기대빈도 = 관측빈도 합계 × 기준 비율 : 기준이 그대로라면 나왔을 건수
4) 조건 확인 : 기대빈도가 모두 5 이상인가
5) 검정 : `stats.chisquare(f_obs=관측빈도, f_exp=기대빈도)`

**비율이 아니라 건수로 바꾸는 이유** : 카이제곱은 건수로 계산한다. 같은 31% 대 40% 차이라도 200건 중의 차이와 2,000건 중의 차이는 무게가 다르다. 건수로 바꿔야 표본 크기가 반영된다.

| 함수 | 기능 |
|---|---|
| `stats.chisquare(f_obs=관측빈도, f_exp=기대빈도)` | 적합도 검정. f_obs는 관측빈도(observed), f_exp는 기대빈도(expected)다. 두 값은 범주 순서가 같고 합계가 같아야 한다 |


공식 문서 : [scipy.stats.chisquare](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chisquare.html) · [numpy.array](https://numpy.org/doc/stable/reference/generated/numpy.array.html)

In [ ]:
# [예제] 이번 달 불량 200건의 유형별 건수가 과거 비율(스크래치 40% · 치수 25% · 이물 20% · 기타 15%)과 맞는지 검정한다
# 귀무가설(H₀): 이번 달 불량 유형의 모비율은 과거 비율과 같다. (p₁ = 0.40, p₂ = 0.25, p₃ = 0.20, p₄ = 0.15)
# 대립가설(H₁): 적어도 한 불량 유형의 모비율이 과거 비율과 다르다.
# 유의수준 α = 0.05
names = ["스크래치", "치수", "이물", "기타"]

# 1) 관측빈도 : 이번 달 실제로 센 불량 건수
f_obs = np.array([62, 70, 38, 30])
# 2) 기준 비율 : 과거 비율 (합계 1)
ratio = np.array([0.40, 0.25, 0.20, 0.15])
# 3) 기대빈도 = 관측빈도 합계 × 기준 비율 : 과거 비율이 이번 달에도 그대로라면 나왔을 건수
f_exp = f_obs.sum() * ratio
#   f_obs.sum() : 관측빈도 합계(200). 배열에 곱하면 값마다 곱해진다 → 80 · 50 · 40 · 30

fit_table = pd.DataFrame({"관측빈도": f_obs, "이번 달 비율": f_obs / f_obs.sum(), "과거 비율": ratio, "기대빈도": f_exp}, index=names)

# 4) 조건 확인 : 기대빈도가 모두 5 이상인가
print("가장 작은 기대빈도:", f_exp.min())

# 5) 수식으로 χ² 구하기 :범주마다 (관측 − 기대)² ÷ 기대(기여도)를 구해 모두 더한다
fit_table["기여도"] = (fit_table["관측빈도"] - fit_table["기대빈도"]) ** 2 / fit_table["기대빈도"]
#   기여도 = (관측 − 기대)² ÷ 기대 : 범주마다 χ² 에 보탠 양. 열끼리 계산하면 행마다 계산된다
print(fit_table.round(2))

# 6) 함수로 검정 : χ² 와 p값을 한 번에 구한다
fit_result = stats.chisquare(f_obs=f_obs, f_exp=f_exp)
print("χ²:", round(fit_result.statistic, 2), "/ 기여도 합계:", round(fit_table["기여도"].sum(), 2), "/ p값:", round(fit_result.pvalue, 4))

# [결과]
# - 이번 달 비율은 스크래치 0.31 · 치수 0.35 · 이물 0.19 · 기타 0.15 로, 과거 0.40 · 0.25 · 0.20 · 0.15 와 다르게 나왔다
# - 가장 작은 기대빈도 30.0 ≥ 5 → 조건을 만족한다
# - 기여도는 치수 8.00, 스크래치 4.05 가 크다. 기여도 합계 12.15 = χ²
# - χ² = 12.15, p = 0.0069
# - 통계적 결론 : p = 0.0069 < 0.05 이므로 귀무가설을 기각한다
# - 사실적 결론 : 이번 달 불량 유형 비율은 과거와 같다고 보기 어렵다. 치수 불량이 관측 70건으로 기대(50건)보다 20건 많았고, 스크래치는 62건으로 기대(80건)보다 18건 적었다 (n = 200)

가장 작은 기대빈도: 30.0
      관측빈도  이번 달 비율  과거 비율  기대빈도   기여도
스크래치    62     0.31   0.40  80.0  4.05
치수      70     0.35   0.25  50.0  8.00
이물      38     0.19   0.20  40.0  0.10
기타      30     0.15   0.15  30.0  0.00
χ²: 12.15 / 기여도 합계: 12.15 / p값: 0.0069


In [ ]:
# [예제] 열 방출 고장(HDF) 115건의 제품 유형 구성이 생산 계획 비율(L 60% · M 30% · H 10%, 가상 기준)과 같은지 검정한다
# 귀무가설(H₀): 열 방출 고장의 제품 유형 모비율은 계획 비율과 같다. (p_L = 0.6, p_M = 0.3, p_H = 0.1)
# 대립가설(H₁): 적어도 한 유형의 모비율이 계획 비율과 다르다.
# 유의수준 α = 0.05

# 1) 관측빈도 : 열 방출 고장 기록의 유형별 건수
hdf = data.loc[data["HDF"] == 1]                            # 열 방출 고장 기록만 고른다
hdf_obs = hdf["Type"].value_counts().loc[["L", "M", "H"]]

print(hdf_obs)

#   value_counts 는 건수가 많은 순서로 나온다. .loc[["L", "M", "H"]] 로 계획 비율과 같은 순서로 맞춘다
# 2) 기준 비율 · 3) 기대빈도
plan = np.array([0.6, 0.3, 0.1])                           # 계획 비율 (L · M · H 순서)
hdf_exp = hdf_obs.sum() * plan                             # 관측빈도 합계(115) × 계획 비율
print(pd.DataFrame({"관측빈도": hdf_obs, "기대빈도": hdf_exp}).round(1))

# 4) 조건 확인
print("가장 작은 기대빈도:", round(hdf_exp.min(), 1))

# 5) 적합도 검정
hdf_result = stats.chisquare(f_obs=hdf_obs, f_exp=hdf_exp)
print("χ²:", round(hdf_result.statistic, 2), "/ p값:", round(hdf_result.pvalue, 4))

# [결과]
# - 관측빈도 L 76 · M 31 · H 8, 기대빈도 69.0 · 34.5 · 11.5, 가장 작은 기대빈도 11.5 ≥ 5
# - χ² = 2.13, p = 0.3447
# - 통계적 결론 : p = 0.3447 ≥ 0.05 이므로 귀무가설을 기각하지 못한다
# - 사실적 결론 : 열 방출 고장의 제품 유형 구성이 계획 비율과 다르다고 볼 통계적 근거가 부족하다.

Type
L    76
M    31
H     8
Name: count, dtype: int64
      관측빈도  기대빈도
Type            
L       76  69.0
M       31  34.5
H        8  11.5
가장 작은 기대빈도: 11.5
χ²: 2.13 / p값: 0.3447


In [ ]:
# [실습]
# 상황 : 생산 계획상 제품 유형 구성은 L 60% · M 30% · H 10% 다(가상 기준). 고장 난 운전 339건의 유형 구성이 이 계획 비율과 같은지 검정한다.
#        귀무가설(H₀): 고장 운전의 제품 유형 모비율은 계획 비율과 같다. (p_L = 0.6, p_M = 0.3, p_H = 0.1)
#        대립가설(H₁): 적어도 한 유형의 모비율이 계획 비율과 다르다.
#        유의수준 α = 0.05
# 고장 운전의 유형별 관측빈도(L · M · H 순서), 기대빈도, 기여도를 표로 구하고, 적합도 검정의 χ² · p값을 구하시오.



## 3. 정리 : 적합도 검정

**결과를 읽는 순서** : 관측빈도 → 기대빈도(모두 5 이상인가) → χ² · p값 → 기여도가 큰 범주 → 두 줄 결론

| 항목 | 내용 |
|---|---|
| 질문 | 한 범주형 변수의 비율이 기준 비율과 같은가 |
| 함수 | `stats.chisquare(f_obs=관측빈도, f_exp=기대빈도)` |
| 기대빈도 | 관측빈도 합계 × 기준 비율 (균등이면 합계 ÷ 범주 수) |
| 조건 | 관측 독립, 한 건은 한 범주에만, 기대빈도 모두 5 이상, 관측과 기대의 합계가 같다 |
| 자유도 | 범주 수 − 1 (함수가 자동 반영) |

### 해석 시 유의점

| 항목 | 해석 기준 |
|---|---|
| 입력 | 비율이 아니라 건수를 넣는다. 기대빈도는 소수여도 된다 |
| 기각했을 때 | 기여도가 큰 범주를 찾아 '어느 범주가 기대보다 많고 적은지'를 함께 쓴다 |


공식 문서 : [scipy.stats.chisquare](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chisquare.html)